In [ ]:
import numpy as np
import pylab as plt
import torch
import torch.nn as nn
from mmengine import Config
from AI4AO.DAO_Utils import *
from AI4AO import PhaseDataset, imshow, imshow_multiple
from AI4AO.TorchPropagator import WFS
from tqdm import tqdm
import torch.nn.functional as F

device = 'cuda'

In [ ]:
WFSParams = dict(
    {
        "Nres":36,
        "sampling":120/36,
        "D": 1.5,
        "centralObstruction": 0.,
        "useNoise": True,
        "Modulation": 0,
        "Wavelength": 635e-9,
        "Nphotons": [4.5, 6],  # Log range of number of photons in measurement
        "RON": [1, 3],  # Read-out noise in photons per pixel per frame
        "Substract_Reference": False,  # Substract or not the reference intensity frame
        "Extract_pupils_pad": 6,
        "Center_noise": 2,
        "Pupil_size_noise": 0.05,  # fraction of nominal pupil size
        "Bin_factor": 1
    }
)


## Atmosphere parameters
AtmosParams = dict(
    {
        "r0": [0.02, 0.12],  # Fired parameter range (m)
        "L0": [10, 30.0000],  # Outter scale range (m)
        "Nphases": 16,  # Number of phases in the batch
        "Layers": [5, 10],  # Number of layers in phase range
        "f_slope": 11.0 / 6.0,  # Slope of the spectrum (11/6) is the default
        "Scintillation": False # Use or not scinitillation
    }
)


## Loop parameters
LoopParams = dict(
    {
        "loopFrequency": 500,
        "delayFrames": 2,
        "windSpeedVector": [1, 10],
        "levelOfCorrection": [0.9, 1.],
        "loopGain": [0.2, 0.5],
        "loopLeak": [0.9,1.] 
    }
)

## Atmosphere parameters
DMParams = dict(
    {
        "Nactuator": 15,
    }
)



In [ ]:
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

In [ ]:
# WFS creation
wfs = WFS(WFSParams, device)

In [ ]:
batch = dataset[0]
opd = batch['opd']
psf_frame = wfs.GetPSF(opd=opd, fov=20, sampling=3)*0
for i in range(100):
    batch = dataset[i]
    opd = batch['opd']
    psf_frame += wfs.GetPSF(opd=opd, fov=20, sampling=3)
imshow(psf_frame.sqrt())
plt.show()

In [ ]:
def GenerateSateliteImage(input: torch.tensor, number_of_squares:int = 5):
    C,W,H = input.shape
    x = torch.linspace(-W//2, W//2, W, device=input.device, dtype=input.dtype)
    y = torch.linspace(-H//2, H//2, H, device=input.device, dtype=input.dtype)
    x,y = torch.meshgrid(x,y)
    angles = torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype)*2*torch.pi
    translation_x = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 3
    translation_y = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 3
    scale_x = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 2
    scale_y = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype) - 0.5) / 2
    brightness = (torch.rand((number_of_squares, C,1,1), device=input.device, dtype=input.dtype)) * scale_x * scale_y
    translation_phases = torch.exp(1j*2*torch.pi*(x * translation_x + y * translation_y))
    sinc_x = torch.sinc(torch.cos(angles)*x*scale_x + torch.sin(angles)*y*scale_y)
    sinc_y = torch.sinc(-torch.sin(angles)*x*scale_x + torch.cos(angles)*y*scale_y)
    PSD = sinc_x * sinc_y * translation_phases * brightness
    return torch.abs(torch.fft.fftshift(torch.fft.fft2(PSD))).sum(dim=0)

In [ ]:
test = torch.zeros((psf_frame.shape[0], 100, 100), device=device, dtype=torch.float32)

test = GenerateSateliteImage(test, 10)
imshow(test)
plt.show()

In [ ]:
conv_test = torch.conv2d(test.unsqueeze(0), psf_frame.unsqueeze(1), groups=psf_frame.shape[0], padding="same")
imshow(conv_test)
plt.show()

In [ ]:

# batch = dataset[0]
opd = batch['opd']
psf_frame = wfs.GetPSF(opd=opd, fov=20, sampling=2)*0
for i in range(30):
    batch = dataset[10*i]
    opd = batch['opd']
    wfs.SetPhotonsAndRON(batch['nphotons']/1, batch['ron'])
    psf_frame += wfs.GetPSF(opd=opd, fov=20, sampling=2)

test = torch.zeros_like(psf_frame)
satelite_GT = GenerateSateliteImage(test, 10)

satelite_blured = torch.conv2d(satelite_GT.unsqueeze(0), psf_frame.unsqueeze(1), groups=psf_frame.shape[0], padding="same")
satelite_blured /= satelite_blured.sum(dim = (-2,-1), keepdim=True)
satelite_blured = wfs.AddNoiseToFrame(satelite_blured)

imshow_multiple([
    {"tensor": psf_frame.sqrt()},
    {"tensor": satelite_GT},
    {"tensor": satelite_blured},
    ],
    )
plt.show()

In [ ]:
def make_satelite_frame(dataset, wfs):
    batch = dataset[0]
    opd = batch['opd']
    psf_frame = wfs.GetPSF(opd=opd, fov=20, sampling=2)
    for i in range(20):
        batch = dataset[10*i]
        opd = batch['opd']
        wfs.SetPhotonsAndRON(batch['nphotons']/1, batch['ron'])
        psf_frame += wfs.GetPSF(opd=opd, fov=20, sampling=2)

    psf_frame /= psf_frame.mean(dim = (-2,-1), keepdim=True)
    test = torch.zeros_like(psf_frame)
    satelite_GT = GenerateSateliteImage(test, 10)
    satelite_GT /= satelite_GT.mean(dim = (-2,-1), keepdim=True)

    satelite_blured = torch.conv2d(satelite_GT.unsqueeze(0), psf_frame.unsqueeze(1), groups=psf_frame.shape[0], padding="same")
    satelite_blured /= satelite_blured.sum(dim = (-2,-1), keepdim=True)
    # satelite_blured = wfs.AddNoiseToFrame(satelite_blured)
    satelite_blured /= satelite_blured.mean(dim = (-2,-1), keepdim=True)

    psf_frame = psf_frame.sqrt()
    psf_frame /= psf_frame.mean(dim = (-2,-1), keepdim=True)

    return psf_frame, satelite_GT, satelite_blured.permute(1,0,2,3)

In [ ]:
def _group_norm_groups(channels, max_groups=8):
    """Largest group count in {8,4,2,1} that evenly divides channels, so
    ConvBlock stays valid for any base_channels the growth knobs are set to."""
    for g in (max_groups, 4, 2, 1):
        if channels % g == 0:
            return g
    return 1


class ConvBlock(nn.Module):
    """Two 3x3 convs (GroupNorm + SiLU each) at a fixed channel count."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(_group_norm_groups(out_channels), out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(_group_norm_groups(out_channels), out_channels),
            nn.SiLU(),
        )

    def forward(self, x):
        return self.block(x)


class DecoderHead(nn.Module):
    def __init__(self, bottleneck_channels, depth, enc_channels):
        super().__init__()

        self.up_blocks = nn.ModuleList([
            ConvBlock(
                (bottleneck_channels if i == depth - 1 else enc_channels[i + 1]) + enc_channels[i],
                enc_channels[i],
            )
            for i in reversed(range(depth))
        ])

        self.head = nn.Conv2d(enc_channels[0], 1, kernel_size=1)

    def forward(self, x, skips):
        for block, skip in zip(self.up_blocks, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = block(torch.cat([x, skip], dim=1))
        
        raw = F.softplus(self.head(x))              # (Nphases,1,H,W), non-negative
        frame = raw / raw.mean(dim=(-2, -1), keepdim=True)
        return frame

class FrameDenoiser(nn.Module):
    """FiLM-conditioned encoder-decoder mapping a stack of n_frames consecutive
    noisy raw WFS detector frames (oldest first, last = current) back toward
    the current frame's noiseless counterpart.

    `depth`/`base_channels`/`n_frames` are the intended growth knobs: increase
    any of them to scale the network's spatial or temporal capacity without
    changing anything else about the architecture.
    """

    def __init__(self, depth=3, base_channels=16, output_frames = 1):
        super().__init__()
        enc_channels = [base_channels * 2 ** i for i in range(depth)]
        bottleneck_channels = enc_channels[-1] * 2

        self.down_blocks = nn.ModuleList([
            ConvBlock(1 if i == 0 else enc_channels[i - 1], enc_channels[i])
            for i in range(depth)
        ])
        self.pool = nn.AvgPool2d(2)

        self.bottleneck = ConvBlock(enc_channels[-1], bottleneck_channels)

        self.decoder_frame = DecoderHead(bottleneck_channels, depth, enc_channels)
        self.decoder_psf = DecoderHead(bottleneck_channels, depth, enc_channels)

    def forward(self, frame_stack, nphotons = torch.tensor([1]).cuda(), ron=torch.tensor([1]).cuda()):
        """frame_stack: (Nphases, n_frames, H, W), oldest first."""
        cond = torch.stack([torch.log10(nphotons).flatten(), ron.flatten()], dim=-1)

        h = frame_stack
        skips = []
        for block in self.down_blocks:
            h = block(h)
            skips.append(h)
            h = self.pool(h)

        h = self.bottleneck(h)

        frame = self.decoder_frame(h, skips)
        psf = self.decoder_psf(h, skips)
        return frame.squeeze(1), psf.squeeze(1)                      # (Nphases,H,W), matches wfs frame convention

In [ ]:
unet = FrameDenoiser(depth=3, base_channels=128).to(device)


n_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,} (budget: 1,000,000)")

In [ ]:
loss_trackers = {}
steps = 10000

optimizer = torch.optim.AdamW(unet.parameters(), 1e-4, fused=(device == 'cuda'))
loss_tracker = torch.zeros(steps, device=device)

unet.train()
progressBar = tqdm(range(steps))

for step in progressBar:
    with torch.no_grad():
        psf, satelite_GT, satelite_blured = make_satelite_frame(dataset, wfs)

    # deconvolved_frame = unet(satelite_blured)
    deconvolved_frame, extracted_psf = unet(satelite_blured)

    reconvolved_frame = torch.conv2d(deconvolved_frame.unsqueeze(0), extracted_psf.square().unsqueeze(1), groups=psf_frame.shape[0], padding="same")
    reconvolved_frame /= reconvolved_frame.mean(dim = (-2,-1), keepdim=True)
    reconvolved_frame = reconvolved_frame.squeeze()

    loss = F.mse_loss(deconvolved_frame, satelite_GT) + F.mse_loss(extracted_psf, psf) / 2 + F.mse_loss(reconvolved_frame, satelite_blured.squeeze()) / 10

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    loss_tracker[step] = loss.detach()

    if step % 100 == 1:
        lower = max(0, step - 100)
        progressBar.set_postfix({'Loss': float(loss_tracker[lower:step].mean())})

    


In [ ]:
torch.save(unet.state_dict(), r"C:\Users\franc\Nextcloud\AI4AO\Data\Deconvolution\43MUNet.pth")

In [ ]:
plt.plot(loss_tracker.cpu())

In [ ]:
F.mse_loss(deconvolved_frame, satelite_GT), F.mse_loss(extracted_psf, psf), F.mse_loss(reconvolved_frame, satelite_blured.squeeze())

In [ ]:
imshow_multiple([
    {"tensor": satelite_blured.squeeze(), "title": "Blury input"},
    {"tensor": satelite_GT.squeeze(), "title": "Perfect image"},
    {"tensor": deconvolved_frame.squeeze(), "title": "Deconvolution"},
    {"tensor": satelite_GT.squeeze() - deconvolved_frame.squeeze(), "title": "Difference", "scale_reference": satelite_GT.squeeze()},
    ],max_channel_number = 9
    )
plt.show()
plt.figure()
imshow_multiple([
    {"tensor": psf.squeeze(), "title": "Real PSF"},
    {"tensor": extracted_psf.squeeze(), "title": "Extracted PSF"},
    {"tensor": psf.squeeze() - extracted_psf.squeeze(), "title": "Difference"},
    ],max_channel_number = 9
    )
plt.show()

In [ ]:
imshow(torch.stack((satelite_blured.squeeze(), reconvolved_frame, satelite_blured.squeeze() - reconvolved_frame), dim = 0))
plt.show()